In [17]:
#imports
import numpy as np
import matplotlib.pyplot as plt
from astropy.table import Table   
from astropy.coordinates import SkyCoord     
from astropy import units as u                   
from astropy.cosmology import Planck18 as cosmo

In [18]:
#read in the catalog
path_to_catalog = '/fs/ess/PCON0003/desi-visualizations/catalogs/zall-tilecumulative-fuji.fits'  # you should not need to change this. But feel free to add other DESI catalogs to this folder and try reading them in. I added the BGS-BRIGHT file from the full DR1.
catalog = Table.read(path_to_catalog)

In [19]:
# trim catalog
RA_max = 360   # RA ranges from 0-360
DEC_max = 90   # DEC ranges from -90 to 90 (although DESI is in the north, so you won't see many below 0)
catalog = catalog[(catalog['TARGET_RA'] < RA_max) & (catalog['TARGET_DEC'] < DEC_max)]

# limit redshift
z_min = 0.01 # below this, they're mostly stars or errors.
z_max = 3    # DESI goes higher than this, but it is sparse and harder to visualize. You can increase this if you want to see more the high-z objects.
catalog = catalog[(catalog['Z'] > z_min) & (catalog['Z'] < z_max)]

# remove galaxies with unrealisting imaging fluxes
flux_g_max = 50  # these are quite faint. You calso can't have negative flux!
flux_g_min = 0
flux_r_max = 50  
flux_r_min = 0
flux_z_max = 50  
flux_z_min = 0
flux_W1_max = 50  
flux_W1_min = 0
flux_W2_max = 50  
flux_W2_min = 0
catalog = catalog[(catalog['FLUX_G'] < flux_g_max) & (catalog['FLUX_G'] > flux_g_min) & (catalog['FLUX_R'] < flux_r_max) & (catalog['FLUX_R'] > flux_r_min) & (catalog['FLUX_Z'] < flux_z_max) & (catalog['FLUX_Z'] > flux_z_min) & (catalog['FLUX_W1'] < flux_W1_max) & (catalog['FLUX_W1'] > flux_W1_min) & (catalog['FLUX_W2'] < flux_W2_max) & (catalog['FLUX_W2'] > flux_W2_min)]

In [20]:
comoving_distance = cosmo.comoving_distance(catalog['Z'])  # calculate the comoving distance to each object based on its redshift and the cosmology (to know: what is comoving distance?)
coords = SkyCoord(ra=catalog['TARGET_RA']*u.degree, dec=catalog['TARGET_DEC']*u.degree, distance=comoving_distance)  # astropy converts this into a 3D coordinate system

In [21]:
#define 3d coords
x = coords.cartesian.x  # these are still astropy objects, i.e. they have assigned units (Mpc)
y = coords.cartesian.y
z = coords.cartesian.z

In [22]:
#define redshift and fluxes
redshift = catalog['Z'] 
flux_g = catalog['FLUX_G']
flux_r = catalog['FLUX_R']
flux_z = catalog['FLUX_Z']
flux_w1 = catalog['FLUX_W1']
flux_w2 = catalog['FLUX_W2']
#xtra
object_type = catalog['SPECTYPE']
zerr = catalog['ZERR']
zwarn = catalog['ZWARN']

In [23]:
#define apparent magnitudes
app_mag_G = -2.5*np.log10(catalog['FLUX_G'])+22.5 
app_mag_R = -2.5*np.log10(catalog['FLUX_R'])+22.5 
app_mag_Z = -2.5*np.log10(catalog['FLUX_Z'])+22.5 
app_mag_W1 = -2.5*np.log10(catalog['FLUX_W1'])+22.5
app_mag_W2 = -2.5*np.log10(catalog['FLUX_W2'])+22.5


In [24]:
#define absolute magnitudes
def apparent_to_absolute_magnitude(m, z):
    d_L = cosmo.luminosity_distance(z)  
    M = m - 5 * np.log10(d_L.to(u.pc).value) + 5  
    return M
abs_mag_g = apparent_to_absolute_magnitude(app_mag_G, redshift)
abs_mag_r = apparent_to_absolute_magnitude(app_mag_R, redshift)
abs_mag_z = apparent_to_absolute_magnitude(app_mag_Z, redshift)
abs_mag_w1 = apparent_to_absolute_magnitude(app_mag_W1, redshift)
abs_mag_w2 = apparent_to_absolute_magnitude(app_mag_W2, redshift)


In [25]:
catalog.columns

<TableColumns names=('TARGETID','SURVEY','PROGRAM','LASTNIGHT','SPGRPVAL','Z','ZERR','ZWARN','CHI2','COEFF','NPIXELS','SPECTYPE','SUBTYPE','NCOEFF','DELTACHI2','PETAL_LOC','DEVICE_LOC','LOCATION','FIBER','COADD_FIBERSTATUS','TARGET_RA','TARGET_DEC','PMRA','PMDEC','REF_EPOCH','LAMBDA_REF','FA_TARGET','FA_TYPE','OBJTYPE','FIBERASSIGN_X','FIBERASSIGN_Y','PRIORITY','SUBPRIORITY','OBSCONDITIONS','RELEASE','BRICKNAME','BRICKID','BRICK_OBJID','MORPHTYPE','EBV','FLUX_G','FLUX_R','FLUX_Z','FLUX_W1','FLUX_W2','FLUX_IVAR_G','FLUX_IVAR_R','FLUX_IVAR_Z','FLUX_IVAR_W1','FLUX_IVAR_W2','FIBERFLUX_G','FIBERFLUX_R','FIBERFLUX_Z','FIBERTOTFLUX_G','FIBERTOTFLUX_R','FIBERTOTFLUX_Z','MASKBITS','SERSIC','SHAPE_R','SHAPE_E1','SHAPE_E2','REF_ID','REF_CAT','GAIA_PHOT_G_MEAN_MAG','GAIA_PHOT_BP_MEAN_MAG','GAIA_PHOT_RP_MEAN_MAG','PARALLAX','PHOTSYS','PRIORITY_INIT','NUMOBS_INIT','CMX_TARGET','DESI_TARGET','BGS_TARGET','MWS_TARGET','SCND_TARGET','SV1_DESI_TARGET','SV1_BGS_TARGET','SV1_MWS_TARGET','SV1_SCND_TARGET',

In [26]:
table = Table([x, y, z, redshift, zerr, zwarn, object_type, flux_g, flux_r, flux_z, flux_w1, flux_w2, app_mag_G, app_mag_R, app_mag_Z, app_mag_W1, app_mag_W2, abs_mag_g, abs_mag_r, abs_mag_z, abs_mag_w1, abs_mag_w2], names=['x', 'y', 'z', 'redshift', 'zerr', 'zwarn', 'object_type', 'flux_g', 'flux_r', 'flux_z', 'flux_w1', 'flux_w2', 'app_mag_G', 'app_mag_R', 'app_mag_Z', 'app_mag_W1', 'app_mag_W2', 'abs_mag_g', 'abs_mag_r', 'abs_mag_z', 'abs_mag_w1', 'abs_mag_w2'])
table.write('desi_catalog.csv', format='csv', overwrite=True)

In [ ]:
table 

x,y,z,redshift,zerr,zwarn,object_type,flux_g,flux_r,flux_z,flux_w1,flux_w2,app_mag_G,app_mag_R,app_mag_Z,app_mag_W1,app_mag_W2,abs_mag_g,abs_mag_r,abs_mag_z,abs_mag_w1,abs_mag_w2
Mpc,Mpc,Mpc,,,,,,,,,,,,,,,,,,,
float64,float64,float64,float64,float64,int64,bytes6,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float64,float64,float64,float64,float64
1158.5377403186735,536.2489567773151,797.330362002583,0.3735858347286332,6.413991864003812e-05,0,GALAXY,1.0944743,6.029373,17.481907,37.671787,26.158922,22.401985,20.54932,19.393528,18.55996,18.95595,-19.175208014721214,-21.02787296223098,-22.183665198559105,-23.01723377155715,-22.62124339985305
4310.55035915863,1947.1258014835435,2923.9436933299967,2.178970627453811,0.0004936601116482719,4,QSO,1.079995,1.3792899,1.5497224,2.1040833,1.5022969,22.416445,22.150862,22.024364,21.692343,22.058111,-23.82064234138224,-24.086225379712317,-24.212722648389075,-24.54474436164591,-24.178975929028724
2764.6924411105724,1251.0594030616521,1881.9395709511725,1.0719633943199487,0.0001379209878579966,0,GALAXY,0.12613815,0.5987699,4.0551972,21.453215,17.3702,24.747883,23.05685,20.979969,19.171268,19.400488,-19.597834558319988,-21.288866967987957,-23.365748376679363,-25.1744489382028,-24.945229501557293
2247.9130170902445,1013.5595094539096,1533.3486234171735,0.8128516220323703,1.6807570922213318e-05,0,GALAXY,1.0990709,1.8720322,4.074051,7.5806646,5.109232,22.397436,21.819216,20.974934,20.300732,20.72911,-21.209146771258446,-21.787367138690087,-22.63164928895864,-23.305851254290673,-22.877472195452782
3323.9173816524612,1532.1168231237798,2285.2704835589075,1.4155970921016339,0.00022921695493051888,0,QSO,1.225721,2.2074194,2.3505902,8.772016,9.962613,22.27902,21.640287,21.572058,20.142252,20.004066,-22.810950485798898,-23.449683395955148,-23.517913071248117,-24.94771882686335,-25.085904327961984
2567.3533032150876,1162.780748825524,1750.4608263800296,0.9689469710156537,8.326162042336996e-05,4,GALAXY,0.4077917,0.47994173,0.5714468,2.9106405,3.73291,23.473904,23.29703,23.107561,21.340029,21.069881,-20.60148572695418,-20.778359887720782,-20.967828271509845,-22.735360620142657,-23.005507943751056
2795.5159371185714,1255.6458380770164,1898.3543809539394,1.0863473054450004,2.7351675374556627e-05,0,GALAXY,0.6306362,0.76480436,1.7175771,1.4432663,2.1832771,23.000553,22.791124,21.91271,22.101633,21.652227,-21.380846102564334,-21.59027488979578,-22.468689043848514,-22.279766161768435,-22.72917183193445
2764.5985552952106,1253.6428201591025,1888.4116720907516,1.07374686844712,3.474482391917619e-05,0,GALAXY,0.31955296,0.42873517,1.2680627,5.837929,2.9877527,23.738644,23.419527,22.242147,20.584352,21.311638,-20.611523722204304,-20.93064031461153,-22.108019922765827,-23.765814875158405,-23.03852949002657
